In [4]:
import tensorflow as tf
import os

# Check TPU availability
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # Detect TPU
    print("✅ TPU detected:", tpu)
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)  # TPU training strategy
except ValueError:
    strategy = tf.distribute.get_strategy()  # Default strategy for CPU/GPU
    print("⚠️ No TPU detected, using default strategy.")

print("Number of devices:", strategy.num_replicas_in_sync)

⚠️ No TPU detected, using default strategy.
Number of devices: 1


In [5]:
# Dataset Paths
TRAIN_DIR = "/kaggle/input/dog-bread-classifier/data_split_new/train"
TEST_DIR = "/kaggle/input/dog-bread-classifier/data_split_new/test"
IMG_SIZE = (224, 224)  # Standard input size for EfficientNetV2
BATCH_SIZE = 32
AUTOTUNE = tf.data.experimental.AUTOTUNE

# Load Dataset
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    shuffle=True,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Normalize Pixel Values (0-1 range)
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)

# Prefetch to Improve Performance
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print("✅ Dataset Loaded Successfully!")

Found 5444 files belonging to 120 classes.
Found 2277 files belonging to 120 classes.
✅ Dataset Loaded Successfully!


In [6]:
import tensorflow_hub as hub
from tensorflow.keras.applications import EfficientNetV2M
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, concatenate, Lambda
from tensorflow.keras.models import Model

# Define Input Shape
input_layer = Input(shape=(224, 224, 3))

# Load Pretrained EfficientNetV2
efficientnet_base = EfficientNetV2M(include_top=False, weights="imagenet", input_tensor=input_layer)
efficientnet_base.trainable = False  # Freeze EfficientNetV2 layers
efficientnet_features = GlobalAveragePooling2D()(efficientnet_base.output)  # Convert feature maps to vectors

# Load Pretrained ViT Model from TensorFlow Hub
vit_model = hub.KerasLayer("https://tfhub.dev/sayakpaul/vit_b16_fe/1", trainable=False)

# Define Lambda Layer with Explicit Output Shape
def vit_lambda(x):
    return vit_model(x)

vit_features = Lambda(vit_lambda, output_shape=(768,))(input_layer)  # ViT output shape is typically (768,)

# Merge Features from EfficientNetV2 & ViT
merged_features = concatenate([efficientnet_features, vit_features])

# Fully Connected Layers for Classification
x = Dense(512, activation="relu")(merged_features)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

# Output Layer (120 Dog Breeds)
output_layer = Dense(120, activation="softmax")(x)

# Define Hybrid Model
hybrid_model = Model(inputs=input_layer, outputs=output_layer)

# Compile the Model
hybrid_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                     loss="sparse_categorical_crossentropy",
                     metrics=["accuracy"])

# Print Model Summary
hybrid_model.summary()
print("✅ Hybrid Model Successfully Created!")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling_1 (Rescaling)   │ (None, 224, 224, 3)    │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv (Conv2D)        │ (None, 112, 112, 24)   │            648 │ rescaling_1[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_bn                   │ (None, 112, 112, 24)   │             96 │ stem_conv[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_activation           │ (None, 112, 112, 24)   │              0 │ stem_bn[0][0]          │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_conv      │ (None, 112, 112, 24)   │          5,184 │ stem_activation[0][0]  │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_bn        │ (None, 112, 112, 24)   │             96 │ block1a_project_conv[… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_activati… │ (None, 112, 112, 24)   │              0 │ block1a_project_bn[0]… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_add (Add)         │ (None, 112, 112, 24)   │              0 │ block1a_project_activ… │
│                           │                        │                │ stem_activation[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_conv      │ (None, 112, 112, 24)   │          5,184 │ block1a_add[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_bn        │ (None, 112, 112, 24)   │             96 │ block1b_project_conv[… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_activati… │ (None, 112, 112, 24)   │              0 │ block1b_project_bn[0]… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_drop (Dropout)    │ (None, 112, 112, 24)   │              0 │ block1b_project_activ… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_add (Add)         │ (None, 112, 112, 24)   │              0 │ block1b_drop[0][0],    │
│                           │                        │                │ block1a_add[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1c_project_conv 

 Total params: 54,361,644 (207.37 MB)

 Trainable params: 1,211,256 (4.62 MB)

 Non-trainable params: 53,150,388 (202.75 MB)

✅ Hybrid Model Successfully Created!


In [7]:
EPOCHS = 20  # Adjust based on TPU performance

# Train Model within TPU Strategy
with strategy.scope():  # Ensure training happens on TPU
    history = hybrid_model.fit(train_dataset, validation_data=test_dataset, epochs=EPOCHS)

Epoch 1/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 277s 1s/step - accuracy: 0.0328 - loss: 5.0693 - val_accuracy: 0.4906 - val_loss: 3.3509
Epoch 2/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 151s 885ms/step - accuracy: 0.2795 - loss: 3.3260 - val_accuracy: 0.7071 - val_loss: 1.6551
Epoch 3/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 151s 885ms/step - accuracy: 0.5294 - loss: 2.0603 - val_accuracy: 0.7861 - val_loss: 0.9345
Epoch 4/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 151s 883ms/step - accuracy: 0.6295 - loss: 1.4385 - val_accuracy: 0.8155 - val_loss: 0.7301
Epoch 5/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 151s 884ms/step - accuracy: 0.6870 - loss: 1.1818 - val_accuracy: 0.8327 - val_loss: 0.6313
Epoch 6/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 150s 880ms/step - accuracy: 0.7316 - loss: 0.9684 - val_accuracy: 0.8322 - val_loss: 0.6020
Epoch 7/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 151s 885ms/step - accuracy: 0.7609 - loss: 0.8655 - val_accuracy: 0.8476 - val_loss: 0.5605
Epoch 8/20
171/171 ━━━━━━━━━━━━━━━━━━━━ 151s 884ms/step - accuracy: 0.7804 - lo

In [13]:
# Save Model
MODEL_PATH = "hybrid_dog_breed_classifier.keras"
hybrid_model.save(MODEL_PATH)
print(f"✅ Model Training Completed & Saved at {MODEL_PATH}")

✅ Model Training Completed & Saved at hybrid_dog_breed_classifier.keras


In [1]:
import tensorflow as tf
import numpy as np
import os
import tensorflow_hub as hub
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Lambda

# Set image path manually
IMAGE_PATH = r"dataset\test\affenpinscher\5a533f3bae76091d6866fa60ba2ec9d4.jpg"

# Load the trained model in .keras format
MODEL_PATH = "hybrid_dog_breed_classifier.keras"
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={"vit_lambda": Lambda(lambda x: hub.KerasLayer("https://tfhub.dev/sayakpaul/vit_b16_fe/1")(x))}
)

# Define Class Labels (120 Breeds)
CLASS_NAMES = sorted(os.listdir("dataset/train"))

# Function to preprocess the image
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))  # Resize to model input shape
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array /= 255.0  # Normalize (0-1 range)
    return img_array

# Function to predict breed
def predict_breed(img_path):
    img_array = preprocess_image(img_path)
    predictions = model.predict(img_array)
    predicted_class_index = np.argmax(predictions[0])  # Get the class with highest probability
    predicted_breed = CLASS_NAMES[predicted_class_index]  # Get class label
    confidence = np.max(predictions[0]) * 100  # Get confidence score
    return predicted_breed, confidence

# Run prediction
if os.path.exists(IMAGE_PATH):
    breed, confidence = predict_breed(IMAGE_PATH)
    print(f"✅ Predicted Breed: {breed} (Confidence: {confidence:.2f}%)")
else:
    print(f"⚠️ Error: Image '{IMAGE_PATH}' not found!")

e:\Softwares\anaconda3\envs\kaggle\lib\site-packages\keras\src\layers\layer.py:393: UserWarning: `build()` was called on layer 'lambda', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
e:\Softwares\anaconda3\envs\kaggle\lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 77s 77s/step
✅ Predicted Breed: affenpinscher (Confidence: 99.96%)
